In [ ]:
### Importing necessary libraries
import numpy as np
import networkx as nx
import xgi
import EoN
import matplotlib.pylab as pylab
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import polars as pl
from joblib import Parallel, delayed
import argparse
from statsmodels.nonparametric.smoothers_lowess import lowess
from src import *

def set_fonts(extra_params={}):
    params = {
        "font.family": "Serif",
        # "font.sans-serif": ["Tahoma", "DejaVu Sans", "Lucida Grande", "Verdana"],
        "mathtext.fontset": "cm",
        "legend.fontsize": 16,
        "legend.title_fontsize": 14,
        "axes.labelsize": 28,
        "axes.titlesize": 28,
        "xtick.labelsize": 20,
        "ytick.labelsize": 20,
        "figure.titlesize": 20,
    }
    for key, value in extra_params.items():
        params[key] = value
    pylab.rcParams.update(params)

set_fonts()

In [ ]:
### Defining the range for eps and del
eps = np.linspace(0.1, 3, 100)
dels = np.linspace(0.1, 3, 100)

### Empty list to store results from the loop
results = []

### Number of nodes
N = 10000
k = 10

### Setting R0 based on the simplest case (BC with well-mixed interactions)
joint_dist1 = create_comm(0.8)
joint_dist2 = create_comm(-0.8)

R0 = 1.1
a = eps @ joint_dist1 @ dels.T
b = eps @ joint_dist2 @ dels.T

gam = 0.1
beta = IBC_beta(a, b, R0, gam, k, 0, 0.5)

args = {
    'N': N,
    'k': k,
    'beta': beta,
    'gam': gam
}

def run_epidemic(e, r, real, args):
    ### Redefining params
    N = args['N']
    k = args['k']
    beta = args['beta']
    gam = args['gam']

    ### Finding community sizes
    size_comm1 = int(r*N)
    size_comm2 = int(N - size_comm1)
    
    ### Creating the distributions and population
    _, comm1 = create_comm(0.8, samplesize=size_comm1)
    _, comm2 = create_comm(-0.8, samplesize=size_comm2)

    ### Store the eps and dels for each individual in a population array
    population = np.concatenate([comm1, comm2], axis = 0)
    
    ### Create the graph
    G = uniform_HPPM(N, 2, k, e, r)
    G = xgi.convert.to_graph(G)
    G = nx.DiGraph(G)

    ### Assign each edge the transmission probability
    e_attr=dict()
    for i,j in G.edges:
        e_attr[(i,j)] = {'weights':population[i][1]*population[j][0]}
    nx.set_edge_attributes(G, e_attr)

    ### Seed the initial infected as 5 percent of the whole population
    first_infected = np.random.choice(range(0, N), size = int(N*0.05), replace = False)
    #first_infected = np.random.choice(range(size_comm1, N), size = int(size_comm2*frac_inf), replace = False)

    ### Run EoN.fast_SIR
    sim = EoN.fast_SIR(G, beta, gam, initial_infecteds=first_infected, transmission_weight = "weights", return_full_data=True)

    ### Find nodes which were in comm1 and comm2
    node_list_comm1 = list(range(0, size_comm1))
    node_list_comm2 = list(range(size_comm1, size_comm1+size_comm2))

    ### Extract data from simulation summary
    _, D1 = sim.summary(node_list_comm1)
    _, D2 = sim.summary(node_list_comm2)

    ### return the results
    return {'real': real, 'e': e, 'r': r, 'R1': D1['R'][-1]/size_comm1, 'R2': D2['R'][-1]/size_comm2, 'Rt': (D1['R'][-1]+D2['R'][-1])/N}

n_cores = 64

results = Parallel(n_jobs = n_cores)(

    delayed(run_epidemic)(e, r, real, args)
    for e in np.linspace(-1, 1, 101)
    for r in np.linspace(0.01, 0.99, 51)
    for real in range(100)

)

### Create a csv file to store results
pl.DataFrame(results).write_csv("../Data/Fig5_data_wBP.csv")

In [ ]:
filepath = '../Data/Fig5_data_wBP.csv'

results_fig4 = pl.read_csv(filepath)
results_Rt = results_fig4.group_by(["e", "r"]).agg(pl.mean("R1")).sort(["e","r"]).pivot(on="e",index="r",values="R1")

fig = plt.figure(figsize=(24.5, 8))
gs = gridspec.GridSpec(1, 2, width_ratios=[6, 1], wspace=0)

ax_heat = fig.add_subplot(gs[0])  # heatmap
ax_marg = fig.add_subplot(gs[1])  # marginal connected to the right

data_matrix = results_Rt.drop("r")
y_labels = ["0.01", *["" for i in range(49)], "0.99"]#np.round(results_Rt["r"].to_list(), 2) # Convert the column to a list for yticklabels
x_labels = ["-1", *["" for i in range(49)], 0, *["" for i in range(49)], "1"]#np.round([float(i) for i in data_matrix.columns], 2) # Column names are used for xticklabels
sns.heatmap(data_matrix, xticklabels=x_labels, yticklabels=y_labels, cmap="rocket_r", ax = ax_heat, vmin = 0, vmax = 0.5, cbar_kws={'label': r'Ratio of $g_2$ infected, $\rho_{g_1}$', 'location':"left", "pad": 0.07})

ax_heat.set_aspect('equal', adjustable='box')
ax_heat.set_ylim(ax_heat.get_ylim()[::-1])

x = np.arange(data_matrix.shape[1]) + 0.5
y = np.arange(data_matrix.shape[0]) + 0.5
cs = ax_heat.contour(x,y,data_matrix, colors="black", linestyles='--')
ax_heat.clabel(cs, inline=True)
ax_heat.tick_params(left=False, bottom=False)

ax_heat.set_xlabel(r"Imbalance parameter, $e$", labelpad=-10)
ax_heat.set_ylabel(r"Ratio of nodes in $g_1$, $r$", labelpad=-20)
plt.setp(ax_heat.get_yticklabels(), rotation=0)

ax_marg.plot(data_matrix['-0.98'] - data_matrix['0.98'], np.linspace(0.01, 0.99, 51), color="#DA3B26", lw=3, label="BP to I")
ax_marg.plot(data_matrix['0.0'] - data_matrix['0.98'], np.linspace(0.01, 0.99, 51), color="#479FF8", lw=3, label = "WM to I")
ax_marg.yaxis.set_visible(False)
ax_marg.set_ylim(0.01, 0.99)
ax_marg.set_xlabel(r"$\Delta \rho_{g_1}$", labelpad=-10)
ax_marg.legend(loc = 'best')

sns.despine(ax=ax_marg, right=True, left=False)
plt.savefig("../Outputs/Fig5_g1.png", dpi = 1000,bbox_inches='tight')
plt.savefig("../Outputs/Fig5_g1.svg", format="svg",bbox_inches='tight')
plt.savefig("../Outputs/Fig5_g1.pdf", format="pdf",bbox_inches='tight')
print(data_matrix.shape)

In [ ]:
filepath = '../Data/Fig5_data_wBP.csv'

results_fig4 = pl.read_csv(filepath)
results_Rt = results_fig4.group_by(["e", "r"]).agg(pl.mean("R2")).sort(["e","r"]).pivot(on="e",index="r",values="R2")

fig = plt.figure(figsize=(24.5, 8))
gs = gridspec.GridSpec(1, 2, width_ratios=[6, 1], wspace=0)

ax_heat = fig.add_subplot(gs[0])  # heatmap
ax_marg = fig.add_subplot(gs[1])  # marginal connected to the right

data_matrix = results_Rt.drop("r")
y_labels = ["0.01", *["" for i in range(49)], "0.99"]#np.round(results_Rt["r"].to_list(), 2) # Convert the column to a list for yticklabels
x_labels = ["-1", *["" for i in range(49)], 0, *["" for i in range(49)], "1"]#np.round([float(i) for i in data_matrix.columns], 2) # Column names are used for xticklabels
sns.heatmap(data_matrix, xticklabels=x_labels, yticklabels=y_labels, cmap="rocket_r", ax = ax_heat, vmin = 0, vmax = 0.5, cbar_kws={'label': r'Ratio of $g_2$ infected, $\rho_{g_2}$', 'location':"left", "pad": 0.07})

ax_heat.set_aspect('equal', adjustable='box')
ax_heat.set_ylim(ax_heat.get_ylim()[::-1])

x = np.arange(data_matrix.shape[1]) + 0.5
y = np.arange(data_matrix.shape[0]) + 0.5
cs = ax_heat.contour(x,y,data_matrix, colors="black", linestyles='--')
ax_heat.clabel(cs, inline=True)
ax_heat.tick_params(left=False, bottom=False)

ax_heat.set_xlabel(r"Imbalance parameter, $e$", labelpad=-10)
ax_heat.set_ylabel(r"Ratio of nodes in $g_1$, $r$", labelpad=-20)
plt.setp(ax_heat.get_yticklabels(), rotation=0)

ax_marg.plot(data_matrix['-0.98'] - data_matrix['0.98'], np.linspace(0.01, 0.99, 51), color="#DA3B26", lw=3, label="BP to I")
ax_marg.plot(data_matrix['0.0'] - data_matrix['0.98'], np.linspace(0.01, 0.99, 51), color="#479FF8", lw=3, label = "WM to I")
ax_marg.yaxis.set_visible(False)
ax_marg.set_ylim(0.01, 0.99)
ax_marg.set_xlabel(r"$\Delta \rho_{g_2}$", labelpad=-10)
ax_marg.legend(loc = 'best')

sns.despine(ax=ax_marg, right=True, left=False)
plt.savefig("../Outputs/Fig5_g2.png", dpi = 1000,bbox_inches='tight')
plt.savefig("../Outputs/Fig5_g2.svg", format="svg",bbox_inches='tight')
plt.savefig("../Outputs/Fig5_g2.pdf", format="pdf",bbox_inches='tight')
print(data_matrix.shape)